# **Muhammad Ibtisam Afzal**

# Assignment No. 12 — Advanced Microsoft Azure AI Services
## Azure AI Content Safety: Text Moderation Demo

**Purpose:** Build a small Python solution that sends text to Azure AI Content Safety and displays the returned harm-category severity scores.

**Before running:** Create an Azure AI Content Safety resource in your Azure subscription. This notebook asks for the resource endpoint and key at runtime. It does not store your secret key in the notebook.

**API:** REST endpoint `POST /contentsafety/text:analyze?api-version=2024-09-01`.

## 1. Service selection and problem solved

**Selected service:** Azure AI Content Safety.

Azure AI Content Safety analyzes user-generated or AI-generated text and detects potentially harmful material across four categories: Hate, Sexual, SelfHarm, and Violence. It returns a severity score for each category. This can help an application moderate text before displaying it, forwarding it to another user, or using it as input to an AI workflow.

**Why this service:** It is suitable for a compact assignment demonstration because a developer can submit a text string through a REST API and inspect structured JSON results.

## 2. Azure resource, endpoint, and authentication

In the Azure Portal:

1. Sign in at https://portal.azure.com.
2. Search for **Azure AI services** or **Content Safety** and create an **Azure AI Content Safety** resource (the exact marketplace wording may vary).
3. Select your subscription, resource group, a supported region, and an available pricing tier; create the resource.
4. Open the deployed resource.
5. Under **Resource Management**, open **Keys and Endpoint** (sometimes shown as **Subscription Key and Endpoint**).
6. Copy the endpoint and one key. Keep the key private.

The program authenticates using the `Ocp-Apim-Subscription-Key` HTTP header. The endpoint and key are entered privately when the notebook runs.

In [1]:
# 3. Install/import dependencies
import sys
!{sys.executable} -m pip -q install requests

import requests
import json
from getpass import getpass
from urllib.parse import urlparse

print("Dependencies are ready.")

Dependencies are ready.


In [2]:
# Enter your Azure resource details securely.
# Paste the endpoint shown on your Azure resource's Keys and Endpoint page.
endpoint = input("Azure Content Safety endpoint: ").strip().rstrip("/")
api_key = getpass("Azure Content Safety key (hidden): ").strip()

if not endpoint.startswith("https://"):
    raise ValueError("The endpoint must start with https://")
if not api_key:
    raise ValueError("API key cannot be empty.")

print("Endpoint accepted:", endpoint)
print("API key received (value hidden).")

Azure Content Safety endpoint: https://navttc-a12.cognitiveservices.azure.com/
Azure Content Safety key (hidden): ··········
Endpoint accepted: https://navttc-a12.cognitiveservices.azure.com
API key received (value hidden).


## 3. Python program to communicate with Azure

The function below sends the text to the Content Safety Text Analysis REST API. It requests all four standard harm categories and uses the four-level severity scale (0, 2, 4, 6).

In [3]:
API_VERSION = "2024-09-01"
ANALYZE_URL = f"{endpoint}/contentsafety/text:analyze?api-version={API_VERSION}"

def analyze_text(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("Please provide non-empty text.")

    headers = {
        "Ocp-Apim-Subscription-Key": api_key,
        "Content-Type": "application/json"
    }
    payload = {
        "text": text,
        "categories": ["Hate", "Sexual", "SelfHarm", "Violence"],
        "outputType": "FourSeverityLevels"
    }

    response = requests.post(
        ANALYZE_URL,
        headers=headers,
        json=payload,
        timeout=60
    )

    # Raise a readable error if Azure returns an unsuccessful status.
    if not response.ok:
        raise RuntimeError(
            f"Azure API request failed ({response.status_code}): {response.text}"
        )

    return response.json()

print("Azure Content Safety request function is ready.")

Azure Content Safety request function is ready.


## 4. Sample input and output demonstration

Run the next cell to send a neutral sample sentence to your actual Azure resource. The JSON shown below the cell will be the live response returned by your resource. **Do not copy an invented output into your submission; capture the result produced by your own run.**

In [4]:
sample_text = (
    "Our university is organizing a technology workshop next week. "
    "Students can register through the official website."
)

print("INPUT TEXT:")
print(sample_text)

result = analyze_text(sample_text)

print("\\nAZURE AI CONTENT SAFETY RESPONSE:")
print(json.dumps(result, indent=2))

print("\\nSEVERITY SUMMARY:")
for item in result.get("categoriesAnalysis", []):
    print(f"{item['category']}: {item['severity']}")

INPUT TEXT:
Our university is organizing a technology workshop next week. Students can register through the official website.
\nAZURE AI CONTENT SAFETY RESPONSE:
{
  "blocklistsMatch": [],
  "categoriesAnalysis": [
    {
      "category": "Hate",
      "severity": 0
    },
    {
      "category": "Sexual",
      "severity": 0
    },
    {
      "category": "SelfHarm",
      "severity": 0
    },
    {
      "category": "Violence",
      "severity": 0
    }
  ]
}
\nSEVERITY SUMMARY:
Hate: 0
Sexual: 0
SelfHarm: 0
Violence: 0


### Understanding the response

The response contains `categoriesAnalysis`. Each entry includes a category and its severity. With `FourSeverityLevels`, the service uses:

| Severity | Meaning |
|---:|---|
| 0 | Safe / no detected severity |
| 2 | Low |
| 4 | Medium |
| 6 | High |

These are model assessment scores, not a guarantee that content is safe or unsafe in every context. An application should define its own moderation thresholds and provide human review for uncertain or high-impact cases.

## 5. Practical real-world use case

**Use case: Moderating comments in an online learning platform.**

When a student submits a discussion-board comment, the platform can send the text to Azure AI Content Safety before publishing it. The application can inspect the returned category scores:

- Low scores may be allowed or logged.
- Medium scores may be held for review, depending on the platform's policy.
- High scores may be blocked or escalated to a human moderator.

This can reduce exposure to abusive or harmful comments while keeping the final moderation decision aligned with the institution's policy.

## 6. One limitation or challenge

**Limitation:** Automated moderation can produce false positives and false negatives. Context, sarcasm, quotation, reclaimed language, and domain-specific wording can affect classification. A severity score should therefore not be treated as a perfect judgment.

**Mitigation:** Set thresholds based on the application's risk tolerance, test with representative examples, monitor outcomes, and provide a human-review path for borderline cases.

## 7. Submission evidence — screenshots to capture

Capture your own screenshots after creating and testing the Azure resource. Add them to your accompanying report (or submit them as separate image files):

1. **Resource overview:** Azure Portal showing the deployed Azure AI Content Safety resource and its name.
2. **Keys and Endpoint page:** Show the endpoint field, but **hide/blur both keys**. Never submit a screenshot exposing a key.
3. **Notebook configuration:** The cell showing that the endpoint was entered and the key was accepted (the key is hidden by `getpass`).
4. **Successful API call:** The notebook cell showing the sample input and the actual JSON response with `categoriesAnalysis`.

If your instructor accepts only a small number of screenshots, prioritize 1, 2 (with keys hidden), and 4.

## 8. Conclusion

This project demonstrates an end-to-end connection to Azure AI Content Safety: creating the Azure resource, configuring endpoint/key authentication, submitting text through Python, reading severity results, and explaining a practical moderation use case and a limitation.

**Security reminder:** Never share your subscription key in screenshots, notebooks, GitHub repositories, or submitted reports. If a key is exposed, regenerate it in Azure Portal.